# Local Friday: Gemma 4 E4B 파인튜닝 및 LiteRT 변환 파이프라인

이 노트북은 `Gemma 4 E4B` 모델을 Local Friday 안드로이드 앱에서 사용할 수 있도록 4-블록 프롬프트에 맞춰 가상 데이터셋으로 파인튜닝(LoRA)하고, 최종적으로 모바일 기기용 `.litertlm` 포맷으로 변환합니다.

In [ ]:
# Step 1: 환경 설정 및 필수 라이브러리 설치
!pip install -q -U keras-nlp keras>=3.0.0
!pip install -q -U ai-edge-litert

import os
# JAX 백엔드를 사용하여 메모리 최적화 및 학습 속도 향상
os.environ["KERAS_BACKEND"] = "jax"
# 메모리 단편화 방지
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.0"

import keras
import keras_nlp
import json

In [ ]:
# Step 2: Kaggle 인증 정보 설정
# Colab 좌측 '열쇠' 아이콘(Secrets)에 KAGGLE_USERNAME과 KAGGLE_KEY를 등록해두면 자동으로 불러옵니다.
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
# Step 3: 가상 데이터셋(Synthetic Dataset) 생성
# Local Friday의 4-블록 시스템 프롬프트와 ActionCard JSON 출력을 모방합니다.

synthetic_data = [
    {
        "instruction": "[시스템 블록]\n당신은 기기 내부에서 동작하는 오프라인 AI 비서 'Local Friday'입니다.\n[메모리 블록]\n과거 대화 없음\n[대화 블록]\nUser: 내일 오후 3시에 치과 예약 일정 잡아줘.\n[현재 입력 블록]\nUser: 내일 오후 3시에 치과 예약 일정 잡아줘.",
        "response": '{"type": "CalendarDraftOutput", "draft": {"title": "치과 예약", "startIso": "2026-06-12T15:00:00Z", "endIso": "2026-06-12T16:00:00Z", "note": "", "confidence": 0.95}}'
    },
    {
        "instruction": "[시스템 블록]\n당신은 기기 내부에서 동작하는 오프라인 AI 비서 'Local Friday'입니다.\n[메모리 블록]\n과거 대화 없음\n[대화 블록]\nUser: 안녕, 넌 누구야?\n[현재 입력 블록]\nUser: 안녕, 넌 누구야?",
        "response": '{"type": "TextOutput", "text": "안녕하세요! 저는 오프라인 환경에서도 당신을 돕는 개인 비서, Local Friday입니다. 무엇을 도와드릴까요?"}'
    }
]

dataset = []
for item in synthetic_data:
    # Gemma Instruct 포맷에 맞춘 템플릿
    prompt = f"<start_of_turn>user\n{item['instruction']}<end_of_turn>\n<start_of_turn>model\n{item['response']}<end_of_turn>"
    dataset.append(prompt)

print(f"생성된 데이터 개수: {len(dataset)}")
print("예시:", dataset[0])

In [ ]:
# Step 4: Keras 모델 로드 및 LoRA 활성화
# 참고: Kaggle에서 제공하는 Keras 포맷의 preset identifier를 사용합니다.
# Gemma 4 E4B의 정확한 preset 이름은 kaggle 모델 페이지의 Keras 탭에서 확인하세요.
PRESET_NAME = "gemma2_instruct_2b_en" # 임시 예시 (Gemma 4 E4B preset으로 교체 필요)

print("모델 다운로드 및 로드 중...")
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(PRESET_NAME)

# LoRA(Low-Rank Adaptation) 활성화
gemma_lm.backbone.enable_lora(rank=16)
gemma_lm.summary()

In [ ]:
# Step 5: 파인튜닝 진행 (Fine-Tuning)
gemma_lm.preprocessor.sequence_length = 512 # 메모리에 맞춰 조절

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=0.01),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

print("학습 시작...")
gemma_lm.fit(dataset, epochs=3, batch_size=1)

In [ ]:
# Step 6: LiteRT (.litertlm) 포맷으로 변환 및 내보내기
import ai_edge_litert

# 먼저 Keras SavedModel 형태로 임시 저장합니다.
export_dir = "gemma_finetuned_keras"
gemma_lm.save(export_dir)

# LiteRT 변환기를 사용하여 변환 (INT4 양자화)
converter = ai_edge_litert.TFLiteConverter.from_saved_model(export_dir)
converter.optimizations = [ai_edge_litert.Optimize.DEFAULT]
# Android 실행을 위한 추가 옵션이 필요할 수 있습니다.
tflite_model = converter.convert()

output_path = "gemma4-e4b-it-custom-q4.litertlm"
with open(output_path, "wb") as f:
    f.write(tflite_model)

print(f"변환 완료! 파일이 저장되었습니다: {output_path}")

In [ ]:
# Step 7: 로컬 환경(PC)으로 파일 다운로드
from google.colab import files
files.download("gemma4-e4b-it-custom-q4.litertlm")